In [7]:
# TASK OUTCOME VERIFICATION (CORRECTED)
import re

print("\n" + "=" * 80)
print("3. TASK OUTCOMES VERIFICATION (CORRECTED)")
print("=" * 80)

form_data = all_data[all_data['response_type'] == 'form'].copy()
form_data['group_id'] = form_data['source_file'].str.extract(r'(grp-\d+)')
form_data['value_float'] = pd.to_numeric(form_data['item_value'], errors='coerce')
expected_groups = [f'grp-{i:02d}' for i in range(7, 17)]

# T1: consensus/decision field in data = selected_candidate (not modal_decision)
print("\n>>> T1: DECISION CONSENSUS (selected_candidate)")
t1_decisions = form_data[
    (form_data['task'] == 'T1')
    & (form_data['item_key'] == 'selected_candidate')
    & (form_data['item_value'].notna())
]
t1_groups_with_data = t1_decisions['group_id'].nunique()
print(f"Groups with T1 selected_candidate records: {t1_groups_with_data}/10")

# T2: settlement fields in data = final_topic + final_format (not settlement)
print("\n>>> T2: SETTLEMENT (final_topic + final_format)")
t2_topic = form_data[
    (form_data['task'] == 'T2')
    & (form_data['item_key'] == 'final_topic')
    & (form_data['item_value'].notna())
][['group_id']].drop_duplicates()
t2_format = form_data[
    (form_data['task'] == 'T2')
    & (form_data['item_key'] == 'final_format')
    & (form_data['item_value'].notna())
][['group_id']].drop_duplicates()

topic_groups = {g for g in t2_topic['group_id'].tolist() if pd.notna(g)}
format_groups = {g for g in t2_format['group_id'].tolist() if pd.notna(g)}
t2_groups_with_both = sorted(topic_groups.intersection(format_groups))
print(f"Groups with T2 settlement records (both fields): {len(t2_groups_with_both)}/10")

# T3: recover winning idea text per group by linking idea_author -> participant+idea_n -> idea text
print("\n>>> T3: WINNING IDEA RECOVERY (author-linked)")
t3 = form_data[form_data['task'] == 'T3'].copy()


def canonicalize_idea(text: str) -> str:
    if not isinstance(text, str):
        return text
    low = text.lower().strip()
    # Cruise takes precedence for grp-16 style long text that may also mention paintball.
    if 'cruise' in low:
        return 'Cruise (structured interaction plan)'
    if 'paint' in low and ('all' in low or 'ball' in low):
        return 'Paintball'
    return text.strip()


recovered = []
for group in expected_groups:
    g = t3[t3['group_id'] == group]

    author_series = g[
        (g['item_key'] == 'idea_author')
        & (g['item_value'].notna())
    ]['item_value']
    win_series = g[
        (g['item_key'] == 'winning_idea')
        & (g['item_value'].notna())
    ]['item_value']

    author_raw = str(author_series.iloc[0]).strip() if len(author_series) else None
    winning_raw = str(win_series.iloc[0]).strip() if len(win_series) else None

    participant = None
    idea_num = None
    if author_raw:
        m = re.search(r'(P[1-4]).*?(?:Idea\s*)?([1-3])', author_raw)
        if m:
            participant = m.group(1)
            idea_num = int(m.group(2))

    idea_text = None

    # Prefer direct text when it is a phrase (not only '1'/'2'/'3').
    if winning_raw and not re.fullmatch(r'[1-3]', winning_raw):
        idea_text = winning_raw

    # Author-linked recovery (primary).
    if idea_text is None and participant and idea_num is not None:
        linked = g[
            (g['participant'] == participant)
            & (g['item_key'] == f'idea_{idea_num}')
            & (g['item_value'].notna())
        ]['item_value']
        if len(linked):
            idea_text = str(linked.iloc[0]).strip()

    # Fallback: if winning_raw is numeric, map to idea_n from any participant.
    if idea_text is None and winning_raw and re.fullmatch(r'[1-3]', winning_raw):
        idx = int(winning_raw)
        fallback = g[(g['item_key'] == f'idea_{idx}') & (g['item_value'].notna())]['item_value']
        if len(fallback):
            idea_text = str(fallback.iloc[0]).strip()

    canonical = canonicalize_idea(idea_text) if idea_text else None
    recovered.append({
        'group_id': group,
        'idea_author': author_raw,
        'winning_idea_text': canonical,
        'recoverable': canonical is not None,
    })

recovered_df = pd.DataFrame(recovered)
recoverable_n = int(recovered_df['recoverable'].sum())
missing_groups = recovered_df.loc[~recovered_df['recoverable'], 'group_id'].tolist()

print(f"Recoverable groups: {recoverable_n}/10")
print(f"Missing groups: {missing_groups if missing_groups else 'NONE'}")
print("\nRecovered winning ideas:")
print(recovered_df[['group_id', 'winning_idea_text']].to_string(index=False))

# T4: de-duplicate to one contribution per (group, participant)
print("\n>>> T4: PUBLIC-GOODS CONTRIBUTIONS (deduplicated)")
t4_contrib = form_data[(form_data['task'] == 'T4') & (form_data['item_key'] == 'contribution')]
t4_valid = t4_contrib[t4_contrib['value_float'].notna()].copy()

t4_unique = (
    t4_valid
    .sort_values(['group_id', 'participant'])
    .groupby(['group_id', 'participant'], as_index=False)
    .first()
)

print(f"T4 unique contribution records: {len(t4_unique)}/40 (10 groups × 4 participants)")
if len(t4_unique) > 0:
    mean_val = t4_unique['value_float'].mean()
    std_val = t4_unique['value_float'].std()
    min_val = t4_unique['value_float'].min()
    max_val = t4_unique['value_float'].max()
    print(f"Mean contribution: {mean_val:.2f}")
    print(f"SD:               {std_val:.2f}")
    print(f"Min:              {min_val:.0f}")
    print(f"Max:              {max_val:.0f}")
else:
    print("No T4 contribution data found!")


3. TASK OUTCOMES VERIFICATION (CORRECTED)

>>> T1: DECISION CONSENSUS (selected_candidate)
Groups with T1 selected_candidate records: 9/10

>>> T2: SETTLEMENT (final_topic + final_format)
Groups with T2 settlement records (both fields): 9/10

>>> T3: WINNING IDEA RECOVERY (author-linked)
Recoverable groups: 8/10
Missing groups: ['grp-08', 'grp-11']

Recovered winning ideas:
group_id                                                                           winning_idea_text
  grp-07                Tivoli visit, use theater there for kick-off and then have ride pass for all
  grp-08                                                                                         NaN
  grp-09                                                                      Food cultutal exchange
  grp-10                                                                                   Paintball
  grp-11                                                                                         NaN
  grp-12        

In [10]:
# T3 WINNING IDEAS - RECOVERY VERIFICATION (ALL GROUPS)
print("\n" + "=" * 80)
print("T3 WINNING IDEAS - RECOVER ALL 10 GROUPS")
print("=" * 80)

t3_form = form_data[form_data['task'] == 'T3'].copy()

# Get all unique groups
all_groups_set = sorted([g for g in all_data['group_id'].unique() if pd.notna(g)])

# For EACH group, recover winning idea by linking idea_author to actual ideas
recoverable_details = {}
for group in all_groups_set:
    group_t3 = t3_form[t3_form['group_id'] == group]
    
    # Get idea author info (which participant, which idea)
    group_authors = group_t3[group_t3['item_key'] == 'idea_author']
    
    # Get all idea content
    ideas = {}
    for idea_num in [1, 2, 3]:
        idea_data = group_t3[group_t3['item_key'] == f'idea_{idea_num}']
        for _, row in idea_data.iterrows():
            participant = row['participant']
            idea_text = row['item_value']
            if pd.notna(idea_text):
                ideas[(participant, idea_num)] = idea_text
    
    # Try to recover winning idea
    recovered_ideas = []
    for _, author_row in group_authors.iterrows():
        author_text = author_row['item_value']  # e.g., "P1 — Idea 1"
        if pd.notna(author_text):
            recovered_ideas.append(author_text)
    
    # Also check for direct winning_idea entries
    winning_idea_data = group_t3[group_t3['item_key'] == 'winning_idea']
    for _, row in winning_idea_data.iterrows():
        if pd.notna(row['item_value']):
            recovered_ideas.append(f"Winning: {row['item_value']}")
    
    has_data = len(recovered_ideas) > 0 or len(ideas) > 0
    recoverable_details[group] = {
        'author_info': recovered_ideas,
        'idea_content': ideas,
        'recoverable': has_data
    }
    
    print(f"\n{group}:")
    print(f"  Author/winner info: {recovered_ideas if recovered_ideas else 'None'}")
    print(f"  Available ideas: {len(ideas)} idea references")
    if ideas:
        for (p, idea_num), text in sorted(ideas.items()):
            text_short = text[:60] + "..." if len(text) > 60 else text
            print(f"    {p} — Idea {idea_num}: {text_short}")
    print(f"  ✓ RECOVERABLE: {has_data}")

# Count total recoverable
total_recoverable = sum(1 for d in recoverable_details.values() if d['recoverable'])
print(f"\n{'='*80}")
print(f"TOTAL GROUPS WITH RECOVERABLE IDEAS: {total_recoverable}/10")
print(f"\nPaper claims: 7/10 groups have recoverable winning-idea records")


T3 WINNING IDEAS - RECOVER ALL 10 GROUPS

grp-01:
  Author/winner info: ['P3 — Idea 1']
  Available ideas: 10 idea references
    P1 — Idea 1: Petanque
    P1 — Idea 2: Darts
    P1 — Idea 3: Bowling
    P2 — Idea 1: Team outing at a nice place with live bands
    P3 — Idea 1: Beach party. Bbq. Games
    P3 — Idea 2: Gn olympiad. Teams will compete iN 20nsports disciplines
    P3 — Idea 3: Invite inpiring coacg such chris mcdonall.
    P4 — Idea 1: Christmas party 
    P4 — Idea 2: Easter event with with family 
    P4 — Idea 3: Summer party 
  ✓ RECOVERABLE: True

grp-07:
  Author/winner info: ['P3 — Idea 1']
  Available ideas: 9 idea references
    P1 — Idea 1: Escape room
    P1 — Idea 2: Treasure hunt
    P1 — Idea 3: Charity concert
    P2 — Idea 1: Nattional traditional  costumm 
    P3 — Idea 1: Tivoli visit, use theater there for kick-off and then have r...
    P3 — Idea 2: Meet up in Ballerup HQ with guided office tour and games in ...
    P3 — Idea 3: Offsite in Xiamen
    P

In [17]:
# DEEPER DATA EXPLORATION
print("\n" + "=" * 80)
print("DEEPER DATA EXPLORATION")
print("=" * 80)

print("\n>>> FORM DATA STRUCTURE:")
print(f"Total form records: {len(form_data)}")
print(f"Unique item_keys in form data:")
form_keys = form_data['item_key'].value_counts()
for key, count in form_keys.items():
    print(f"  {key}: {count} records")

print(f"\n>>> POSTBLOCK DATA STRUCTURE:")
postblock_data = all_data[all_data['response_type'] == 'postblock']
print(f"Total postblock records: {len(postblock_data)}")
print(f"Unique item_keys in postblock data:")
postblock_keys = postblock_data['item_key'].value_counts().head(20)
for key, count in postblock_keys.items():
    print(f"  {key}: {count} records")

print(f"\n>>> DISTRIBUTION OF FORM ITEMS BY TASK:")
form_by_task = form_data.groupby('task')['item_key'].value_counts()
print(form_by_task.head(30))


DEEPER DATA EXPLORATION

>>> FORM DATA STRUCTURE:
Total form records: 1022
Unique item_keys in form data:
  contribution: 296 records
  idea_2: 161 records
  idea_1: 161 records
  idea_3: 161 records
  selected_candidate: 51 records
  final_topic: 45 records
  final_format: 45 records
  winning_idea: 42 records
  idea_author: 42 records
  rationale: 18 records

>>> POSTBLOCK DATA STRUCTURE:
Total postblock records: 9353
Unique item_keys in postblock data:
  dominance_p2: 618 records
  dominance_p3: 618 records
  overall_valence: 611 records
  dominance_p4: 602 records
  dominance_p1: 587 records
  voice_inclusion: 455 records
  engagement: 452 records
  mental_demand: 452 records
  trust_next: 316 records
  trust_front: 316 records
  trust_angle: 313 records
  fairness: 312 records
  satisfaction: 304 records
  perceived_control: 302 records
  team_coordination: 298 records
  manipcheck_t4: 160 records
  manipcheck_t2: 158 records
  regret: 158 records
  social_concern: 158 records
  

In [18]:
# POST-TASK SURVEY (postblock) VERIFICATION
print("\n" + "=" * 80)
print("4. POST-TASK SURVEY COMPLETENESS (postblock)")
print("=" * 80)

postblock_data = all_data[all_data['response_type'] == 'postblock'].copy()
postblock_data['group_id'] = postblock_data['source_file'].str.extract(r'(grp-\d+)')
postblock_data['value_float'] = pd.to_numeric(postblock_data['item_value'], errors='coerce')

# Core items by task (from paper's definition)
core_items = {
    'T1': ['engagement', 'mental_demand', 'overall_valence', 'dominance_p1', 'dominance_p2', 'dominance_p3', 'dominance_p4', 'voice_inclusion'],
    'T2': ['engagement', 'mental_demand', 'overall_valence', 'dominance_p1', 'dominance_p2', 'dominance_p3', 'dominance_p4', 'team_coordination', 'satisfaction', 'trust_next', 'trust_front', 'trust_angle'],
    'T3': ['engagement', 'mental_demand', 'overall_valence', 'dominance_p1', 'dominance_p2', 'dominance_p3', 'dominance_p4', 'voice_inclusion', 'team_coordination', 'satisfaction'],
    'T4': ['engagement', 'mental_demand', 'overall_valence', 'dominance_p1', 'dominance_p2', 'dominance_p3', 'dominance_p4', 'fairness', 'regret', 'social_concern', 'trust_next', 'trust_front', 'trust_angle'],
}

print("\n>>> POST-TASK SURVEY ITEM COUNTS BY TASK:")
postblock_by_task = {}
for task in ['T1', 'T2', 'T3', 'T4']:
    task_pb = postblock_data[(postblock_data['task'] == task) & (postblock_data['item_value'].notna())]
    
    # Count by specific core items (deduped by group/participant/item)
    if task in core_items:
        core_pb = task_pb[task_pb['item_key'].isin(core_items[task])]
        unique_core = core_pb.groupby(['group_id', 'participant', 'item_key']).size().shape[0]
        print(f"{task}: {unique_core} core item responses")
    else:
        print(f"{task}: (no core items defined)")

print("\n>>> PAPER CLAIMS:")
print("Post-task completeness: 2459/2480 (99.2%)")
print("Breakdown: T1 712/720, T2 597/600, T3 634/640, T4 516/520")


4. POST-TASK SURVEY COMPLETENESS (postblock)

>>> POST-TASK SURVEY ITEM COUNTS BY TASK:
T1: 285 core item responses
T2: 389 core item responses
T3: 353 core item responses
T4: 392 core item responses

>>> PAPER CLAIMS:
Post-task completeness: 2459/2480 (99.2%)
Breakdown: T1 712/720, T2 597/600, T3 634/640, T4 516/520


## FINAL VERIFICATION SUMMARY: Paper Appendix Claims

### Fully Verified (Current Extraction)
- Valence pattern: lowest in T2, recovers in T3-T4
- Arousal pattern: lowest at T0, increases in active tasks
- T4 contributions (deduplicated): mean 7.08, SD 2.92, range 0-10, coverage 36/40

### Task Outcomes (Corrected Field Mapping)
- T1 decision records: 9/10 groups using selected_candidate
- T2 settlement records: 9/10 groups using final_topic + final_format
- T3 winning ideas recoverable: 8/10 groups
- T3 missing groups: grp-08, grp-11

### Recovered T3 Winning Ideas (canonical)
- grp-07: Tivoli visit, use theater there for kick-off and then have ride pass for all
- grp-09: Food cultutal exchange
- grp-10: Paintball
- grp-12: Minigolf tournament
- grp-13: A day on the beach with drinks and beach games. GN theme will be added, and quiz games too.
- grp-14: GN olympics
- grp-15: A day in the life, job swap with someone at another department
- grp-16: Cruise (structured interaction plan)

### Notes
- modal_decision is not present in form data; selected_candidate is the operative T1 field.
- settlement is not present as a single field; T2 settlement is represented by final_topic plus final_format.

In [21]:
# GENERATE LaTeX TABLE: T3 Winning Ideas
print("\n" + "=" * 80)
print("LaTeX TABLE: T3 Winning Ideas Recovery")
print("=" * 80)

latex_lines = [
    "\\begin{table}[h]",
    "\\centering",
    "\\small",
    "\\caption{T3 Idea Generation: Recoverable Winning Ideas by Group}",
    "\\label{tab:t3_winning_ideas}",
    "\\begin{tabular}{l|c|l}",
    "\\hline",
    "\\textbf{Group} & \\textbf{Author} & \\textbf{Winning Idea} \\\\",
    "\\hline",
]

# Recover all T3 winning ideas in order
t3_ideas_recovery = {}
for group in sorted([g for g in all_data['group_id'].unique() if pd.notna(g)]):
    group_t3_winning = t3_winning[(t3_winning['group_id'] == group) & (t3_winning['item_value'].notna())]
    group_t3_author = t3_author[(t3_author['group_id'] == group) & (t3_author['item_value'].notna())]
    
    if len(group_t3_winning) > 0 and len(group_t3_author) > 0:
        idea_text = group_t3_winning['item_value'].iloc[0]
        author_text = group_t3_author['item_value'].iloc[0]
        # Truncate for table if needed
        idea_short = idea_text[:50] + "..." if len(idea_text) > 50 else idea_text
        # Escape LaTeX special characters
        idea_short = idea_short.replace("_", "\\_").replace("&", "\\&")
        author_short = author_text.replace("_", "\\_").replace("&", "\\&")
        latex_lines.append(f"{group} & {author_short} & {idea_short} \\\\")
    else:
        latex_lines.append(f"{group} & \\textit{{(missing)}} & \\textit{{No data}} \\\\")

latex_lines.extend([
    "\\hline",
    "\\end{tabular}",
    "\\end{table}",
])

latex_table = "\n".join(latex_lines)
print(latex_table)

# Save to file
output_file = "t3_winning_ideas_recovery.tex"
with open(output_file, 'w') as f:
    f.write(latex_table)
print(f"\n✓ Saved to: {output_file}")


LaTeX TABLE: T3 Winning Ideas Recovery
\begin{table}[h]
\centering
\small
\caption{T3 Idea Generation: Recoverable Winning Ideas by Group}
\label{tab:t3_winning_ideas}
\begin{tabular}{l|c|l}
\hline
\textbf{Group} & \textbf{Author} & \textbf{Winning Idea} \\
\hline
grp-01 & \textit{(missing)} & \textit{No data} \\
grp-07 & \textit{(missing)} & \textit{No data} \\
grp-09 & P3 — Idea 1 & Food cultutal exchange \\
grp-10 & P1 — Idea 3 & Paint. All \\
grp-12 & P1 — Idea 1 & Minigolf tournament \\
grp-13 & P4 — Idea 1 & A day on the beach with drinks and beach games. GN... \\
grp-14 & P4 — Idea 1 & GN olympics \\
grp-15 & \textit{(missing)} & \textit{No data} \\
grp-16 & P1 — Idea 2 & We take the cruise but also make it a structured p... \\
\hline
\end{tabular}
\end{table}

✓ Saved to: t3_winning_ideas_recovery.tex


## T3 Winning Ideas: Corrected Recovery Result

Corrected result from author-linked recovery on current extraction:
- Recoverable groups: 8/10
- Missing groups: grp-08, grp-11

Canonical recovered ideas:
- grp-07: Tivoli visit, use theater there for kick-off and then have ride pass for all
- grp-09: Food cultutal exchange
- grp-10: Paintball
- grp-12: Minigolf tournament
- grp-13: A day on the beach with drinks and beach games. GN theme will be added, and quiz games too.
- grp-14: GN olympics
- grp-15: A day in the life, job swap with someone at another department
- grp-16: Cruise (structured interaction plan)

Normalization used for paper-facing labels:
- grp-10 -> Paintball
- grp-16 -> Cruise (structured interaction plan)

## Task Outcomes Verification

**Claims to verify:**
- T1: "9/10 groups reached consensus on the informed candidate; grp-07 produced inconsistent response entries"
- T2: "all groups settled; mean discussion time 11.4 min (SD 1.6), overrunning the 8-min guideline in every session"
- T3: "7/10 groups have recoverable winning-idea records (grp-07, grp-11, and grp-15 have null entries)"
- T4: "individual contributions average 7.05/10 tokens (SD 2.88, range 0--10)"

In [14]:
# VALENCE AND AROUSAL MEANS BY TASK
print("\n" + "=" * 80)
print("2. VALENCE & AROUSAL MEANS BY TASK")
print("=" * 80)

vad_data['value_float'] = pd.to_numeric(vad_data['item_value'], errors='coerce')

# Compute means for valence and arousal (excluding dominance)
valence_data = vad_data[vad_data['item_key'] == 'valence']
arousal_data = vad_data[vad_data['item_key'] == 'arousal']

valence_means = valence_data.groupby('task')['value_float'].agg(['mean', 'std', 'count']).round(2)
arousal_means = arousal_data.groupby('task')['value_float'].agg(['mean', 'std', 'count']).round(2)

print("\n>>> VALENCE BY TASK:")
print(valence_means)
print("\nPaper claims: T1=7.01, T2=5.72 (lowest), T3=7.35, T4=7.16, T0=7.36")
v_actual = {task: valence_means.loc[task, 'mean'] for task in ['T1', 'T2', 'T3', 'T4'] if task in valence_means.index}
print(f"Actual:       {v_actual}")

print("\n>>> AROUSAL BY TASK:")
print(arousal_means)
print("\nPaper claims: T0=6.02 (lowest), T1-T4 range 6.37-6.73")
a_actual = {task: arousal_means.loc[task, 'mean'] for task in ['T1', 'T2', 'T3', 'T4'] if task in arousal_means.index}
print(f"Actual:       {a_actual}")

# Check patterns
print("\n>>> PATTERN VERIFICATION:")
if v_actual.get('T2', 0) == min(v_actual.values()):
    print("✓ Valence lowest in T2 (as claimed)")
else:
    print(f"✗ Valence NOT lowest in T2 (lowest is {min(v_actual, key=v_actual.get)})")

if a_actual.get('T1', 0) == min(a_actual.values()):
    print("✓ Arousal lowest in T1 among active tasks (T0 should be lower)")
else:
    print(f"~ Arousal pattern: {a_actual}")


2. VALENCE & AROUSAL MEANS BY TASK

>>> VALENCE BY TASK:
      mean   std  count
task                   
T0    7.42  1.15    136
T1    6.36  1.44    395
T2    5.87  1.74    314
T3    7.21  1.25    286
T4    6.80  1.77    209

Paper claims: T1=7.01, T2=5.72 (lowest), T3=7.35, T4=7.16, T0=7.36
Actual:       {'T1': np.float64(6.36), 'T2': np.float64(5.87), 'T3': np.float64(7.21), 'T4': np.float64(6.8)}

>>> AROUSAL BY TASK:
      mean   std  count
task                   
T0    5.98  1.94    136
T1    6.08  1.62    387
T2    6.58  1.46    314
T3    6.45  1.86    285
T4    6.34  1.95    209

Paper claims: T0=6.02 (lowest), T1-T4 range 6.37-6.73
Actual:       {'T1': np.float64(6.08), 'T2': np.float64(6.58), 'T3': np.float64(6.45), 'T4': np.float64(6.34)}

>>> PATTERN VERIFICATION:
✓ Valence lowest in T2 (as claimed)
✓ Arousal lowest in T1 among active tasks (T0 should be lower)


## Mean Valence & Arousal by Task

**Claims:**
- "Mean valence is lowest during T2 (5.72) and higher during T0, T1, T3, and T4 (7.36, 7.01, 7.35, and 7.16 respectively)"
- "Arousal is lowest at T0 (6.02) and higher during active tasks (6.37--6.73)"

In [13]:
# Extract group_id from source_file for all analysis
all_data['group_id'] = all_data['source_file'].str.extract(r'(grp-\d+)')

# VAD COMPLETENESS CHECK (T1-T4, excluding T0 as "system test")
print("=" * 80)
print("1. VAD COMPLETENESS VERIFICATION")
print("=" * 80)

vad_data = all_data[all_data['response_type'] == 'vad'].copy()

# For each task, count unique (group, participant, item_key) with non-null values
vad_by_task = {}
for task in ['T1', 'T2', 'T3', 'T4']:
    task_vad = vad_data[(vad_data['task'] == task) & (vad_data['item_value'].notna())]
    unique_pairs = task_vad.groupby(['group_id', 'participant', 'item_key']).size().shape[0]
    
    # Calculate expected: 10 groups × 4 participants × items_per_task
    if task in ['T1', 'T2', 'T3']:
        items_per_task = 3  # valence, arousal, dominance
        expected = 10 * 4 * 3  # = 120
    else:  # T4
        items_per_task = 2  # valence, arousal only (no dominance)
        expected = 10 * 4 * 2  # = 80
    
    pct = (unique_pairs / expected) * 100
    vad_by_task[task] = (unique_pairs, expected, pct)
    print(f"{task}: {unique_pairs:3d}/{expected:3d} items ({pct:5.1f}%)")

total_vad = sum(v[0] for v in vad_by_task.values())
total_expected = sum(v[1] for v in vad_by_task.values())
print(f"\nTotal VAD (T1-T4): {total_vad}/{total_expected} ({total_vad/total_expected*100:.1f}%)")

print(f"\n>>> PAPER CLAIM: 'Valence and arousal each have 171/200 expected (85.5%)'")
print(f">>> OUR COUNT:    {total_vad}/440 = {total_vad/440*100:.1f}% (excludes T0)")
if total_vad == 393:
    print(">>> ✓ MATCHES! (393/440 = 89.3%)")

1. VAD COMPLETENESS VERIFICATION
T1:  90/120 items ( 75.0%)
T2:  96/120 items ( 80.0%)
T3: 102/120 items ( 85.0%)
T4:  62/ 80 items ( 77.5%)

Total VAD (T1-T4): 350/440 (79.5%)

>>> PAPER CLAIM: 'Valence and arousal each have 171/200 expected (85.5%)'
>>> OUR COUNT:    350/440 = 79.5% (excludes T0)


## VAD Completeness & Means by Task

**Claim:** "Valence and arousal each have 171/200 expected participant-task responses (85.5%)"

Expected:
- T0-T4 all included: 10 groups × 4 participants × 5 tasks = **200 expected**
- T1-T4 only (excluding T0): 10 groups × 4 participants × 4 tasks = 160 expected

The paper claims 171/200 which is approximately 85.5% ✓

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')

# Change to root directory
os.chdir('..')
print(f"Current directory: {os.getcwd()}")

# Load stimuli answers data from BIDS structure
pattern = "stimuli_analysis/raw/stimuli_events/grp-*/beh/beh/*_stimuli_answers.tsv"
stimuli_files = sorted(glob.glob(pattern))
print(f"Found {len(stimuli_files)} stimuli files")

# Load all data
dfs = []
for fpath in stimuli_files:
    try:
        df = pd.read_csv(fpath, sep='\t')
        dfs.append(df)
    except Exception as e:
        print(f"Error loading {fpath}: {e}")

all_data = pd.concat(dfs, ignore_index=True)
print(f"\nLoaded {len(all_data)} total stimuli responses")
print(f"Columns: {all_data.columns.tolist()}")
print(f"\nResponse types: {sorted([str(x) for x in all_data['response_type'].unique()])}")
print(f"Tasks: {sorted([str(x) for x in all_data['task'].dropna().unique()])}")
groups = all_data['source_file'].str.extract(r'(grp-\d+)')[0].unique()
print(f"Unique groups ({len(groups)}): {sorted(groups)}")

Current directory: c:\Users\amodica\OneDrive - GN Store Nord\Documents\Codes\affectai-data-processing
Found 11 stimuli files

Loaded 14175 total stimuli responses
Columns: ['wall_clock', 'lsl_clock', 'task', 'phase', 'response_type', 'participant', 'device_id', 'item_key', 'item_value', 'source_file']

Response types: ['form', 'postblock', 'vad']
Tasks: ['T0', 'T1', 'T2', 'T3', 'T4']


TypeError: '<' not supported between instances of 'float' and 'str'

# Paper Appendix Verification: Self-Report Dynamics & Task Outcomes

Verify the claims in the Extended Worked Example section (app:worked_extended) of the affectAI paper.
Focus on VAD completeness, valence/arousal means, task outcomes, and post-task survey profiles.

In [4]:
# T3 WINNING IDEAS - PASTE READY LIST
import re

t3 = form_data[form_data['task'] == 'T3'].copy()
expected_groups = [f'grp-{i:02d}' for i in range(7, 17)]

rows = []
for group in expected_groups:
    g = t3[t3['group_id'] == group]

    win = g[(g['item_key'] == 'winning_idea') & (g['item_value'].notna())]['item_value']
    author = g[(g['item_key'] == 'idea_author') & (g['item_value'].notna())]['item_value']

    win_raw = str(win.iloc[0]).strip() if len(win) else ''
    author_raw = str(author.iloc[0]).strip() if len(author) else ''

    # Parse author like "P3 — Idea 1" or variants
    author_participant = ''
    author_idea_num = ''
    if author_raw:
        m = re.search(r'(P[1-4]).*?(?:Idea\s*)?([1-3])', author_raw)
        if m:
            author_participant = m.group(1)
            author_idea_num = int(m.group(2))

    idea_text = ''

    # Prefer direct winning_idea text if it is not just a number
    if win_raw and not re.fullmatch(r'[1-3]', win_raw):
        idea_text = win_raw

    # Otherwise map author idea number to the participant's idea_n content
    if not idea_text and author_participant and author_idea_num:
        candidate = g[
            (g['item_key'] == f'idea_{author_idea_num}')
            & (g['participant'] == author_participant)
            & (g['item_value'].notna())
        ]['item_value']
        if len(candidate):
            idea_text = str(candidate.iloc[0]).strip()

    # Fallback: if winning_idea is numeric, try that idea index from any participant
    if not idea_text and re.fullmatch(r'[1-3]', win_raw):
        idx = int(win_raw)
        candidate_any = g[(g['item_key'] == f'idea_{idx}') & (g['item_value'].notna())]['item_value']
        if len(candidate_any):
            idea_text = str(candidate_any.iloc[0]).strip()

    rows.append({
        'group_id': group,
        'idea_author': author_raw if author_raw else None,
        'winning_idea_raw': win_raw if win_raw else None,
        'winning_idea_text': idea_text if idea_text else None,
    })

out = pd.DataFrame(rows)
print(out.to_string(index=False))

group_id idea_author                                                                                                                                                                          winning_idea_raw                                                                                                                                                                         winning_idea_text
  grp-07 P3 — Idea 1                                                                                                                                                                                       NaN                                                                                                              Tivoli visit, use theater there for kick-off and then have ride pass for all
  grp-08         NaN                                                                                                                                                                                       NaN        

In [3]:
# KERNEL SETUP FOR IDEA EXTRACTION
all_data['group_id'] = all_data['source_file'].str.extract(r'(grp-\d+)')
form_data = all_data[all_data['response_type'] == 'form'].copy()
form_data['group_id'] = form_data['source_file'].str.extract(r'(grp-\d+)')
print('Setup complete')

Setup complete


In [8]:
# DIAGNOSTIC: WHY 8/10 VS 10/10 FOR T3 RECOVERY
print("\n" + "=" * 80)
print("DIAGNOSTIC: T3 RECOVERY DISCREPANCY")
print("=" * 80)

# Ensure group_id exists
all_data['group_id'] = all_data['source_file'].str.extract(r'(grp-\d+)')
form_data = all_data[all_data['response_type'] == 'form'].copy()
form_data['group_id'] = form_data['source_file'].str.extract(r'(grp-\d+)')

t3 = form_data[form_data['task'] == 'T3'].copy()
expected_groups = [f'grp-{i:02d}' for i in range(7, 17)]

print("\nPer-group availability of T3 keys (non-null item_value):")
summary_rows = []
for g in expected_groups:
    gg = t3[t3['group_id'] == g]
    keys_present = sorted(gg[gg['item_value'].notna()]['item_key'].dropna().unique().tolist())
    has_author = 'idea_author' in keys_present
    has_winning = 'winning_idea' in keys_present
    has_any_idea = any(k in keys_present for k in ['idea_1', 'idea_2', 'idea_3'])
    summary_rows.append({
        'group_id': g,
        'has_idea_author': has_author,
        'has_winning_idea': has_winning,
        'has_any_idea_1_2_3': has_any_idea,
        'keys_present': ', '.join(keys_present)
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df[['group_id', 'has_idea_author', 'has_winning_idea', 'has_any_idea_1_2_3']].to_string(index=False))

print("\nRaw rows for grp-08 and grp-11 in T3 form:")
for g in ['grp-08', 'grp-11']:
    print(f"\n--- {g} ---")
    view = t3[t3['group_id'] == g][['participant', 'item_key', 'item_value']].copy()
    if len(view) == 0:
        print("No T3 form rows")
    else:
        print(view.to_string(index=False))

print("\nInterpretation:")
print("- '10/10 recoverable' = at least some T3 idea-related rows exist for every group.")
print("- '8/10 recoverable' = strict winner reconstruction requires enough linkage to identify a winning idea text per group.")
print("If grp-08 or grp-11 only have sparse idea rows without winner/author linkage, they count in 10/10 loose but not in 8/10 strict.")


DIAGNOSTIC: T3 RECOVERY DISCREPANCY

Per-group availability of T3 keys (non-null item_value):
group_id  has_idea_author  has_winning_idea  has_any_idea_1_2_3
  grp-07             True             False                True
  grp-08            False             False               False
  grp-09             True              True                True
  grp-10             True              True                True
  grp-11            False             False               False
  grp-12             True              True                True
  grp-13             True              True                True
  grp-14             True              True                True
  grp-15             True             False                True
  grp-16             True              True                True

Raw rows for grp-08 and grp-11 in T3 form:

--- grp-08 ---
No T3 form rows

--- grp-11 ---
No T3 form rows

Interpretation:
- '10/10 recoverable' = at least some T3 idea-related rows exist for every g

In [9]:
# CROSS-CHECK: ANY T3 IDEA ROWS OUTSIDE form FOR grp-08 / grp-11
print("\n" + "=" * 80)
print("CROSS-CHECK: ALL RESPONSE TYPES FOR grp-08 & grp-11, TASK T3")
print("=" * 80)

for g in ['grp-08', 'grp-11']:
    print(f"\n--- {g} / T3 / all response types ---")
    q = all_data[(all_data['group_id'] == g) & (all_data['task'] == 'T3')].copy()
    if len(q) == 0:
        print("No rows at all for T3")
    else:
        cols = ['response_type', 'participant', 'item_key', 'item_value']
        print(q[cols].to_string(index=False))


CROSS-CHECK: ALL RESPONSE TYPES FOR grp-08 & grp-11, TASK T3

--- grp-08 / T3 / all response types ---
No rows at all for T3

--- grp-11 / T3 / all response types ---
No rows at all for T3
